## 简介与设置

如果你已经完成了我的 [从零开始构建线性模型与神经网络](https://www.kaggle.com/code/jhoward/linear-model-and-neural-net-from-scratch) 笔记本，现在正是了解如何用库来实现同样功能的好时机——而不必从零开始。我们将使用 fastai 和 PyTorch。使用这些库的好处在于：

- 最佳实践会自动为你处理——fast.ai 已经进行了数千小时的实验，帮你找到了最优的参数设置
- 减少配置时间，意味着你有更多时间去尝试新想法
- 每一个新想法的实现成本更低，因为 fastai 和 PyTorch 会替你处理大量繁琐的细节
- 你随时可以从 fastai 降级到 PyTorch（如果需要自定义某些部分），或者从 fastai 的应用层 API 降级到 fastai 的中层或底层 API，甚至可以从 PyTorch 降级到纯 Python 进行深度定制。

让我们来看看实际效果如何。我们将从与"从零开始"笔记本相同的库配置开始：

In [ ]:
from pathlib import Path
import os

iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')
if iskaggle:
    path = Path('../input/titanic')
    !pip install -Uqq fastai
else:
    import zipfile,kaggle
    path = Path('titanic')
    kaggle.api.competition_download_cli(str(path))
    zipfile.ZipFile(f'{path}.zip').extractall(path)

我们将导入 fastai 表格库，设置随机种子以使笔记本可复现，并选择一个合理的有效数字位数以在表格中显示：

In [ ]:
from fastai.tabular.all import *

pd.options.display.float_format = '{:.2f}'.format
set_seed(42)

## 准备数据

我们将像之前一样读取 CSV 文件：

In [ ]:
df = pd.read_csv(path/'train.csv')

当你从头开始做所有事情时，每一个特征工程都需要大量的工作，因为你必须考虑诸如虚拟变量、归一化、缺失值等问题。但使用 fastai，这些都已为你处理好了。所以让我们大胆地创建大量新特征！我们将使用这个精彩的 [Titanic 特征工程笔记本](https://www.kaggle.com/code/gunesevitan/titanic-advanced-feature-engineering-tutorial/) 中一些最有趣的特征（如果你喜欢的话，请务必点击该链接并为该笔记本点赞，以感谢作者的辛勤付出！）

In [ ]:
def add_features(df):
    df['LogFare'] = np.log1p(df['Fare'])
    df['Deck'] = df.Cabin.str[0].map(dict(A="ABC", B="ABC", C="ABC", D="DE", E="DE", F="FG", G="FG"))
    df['Family'] = df.SibSp+df.Parch
    df['Alone'] = df.Family==1
    df['TicketFreq'] = df.groupby('Ticket')['Ticket'].transform('count')
    df['Title'] = df.Name.str.split(', ', expand=True)[1].str.split('.', expand=True)[0]
    df['Title'] = df.Title.map(dict(Mr="Mr",Miss="Miss",Mrs="Mrs",Master="Master")).value_counts(dropna=False)

add_features(df)

正如我们在上一个笔记本中所讨论的，我们可以使用 `RandomSplitter` 来分离训练集和验证集：

In [ ]:
splits = RandomSplitter(seed=42)(df)

现在，准备训练数据的整个过程只需这一个单元格！：

In [ ]:
dls = TabularPandas(
    df, splits=splits,
    procs = [Categorify, FillMissing, Normalize],
    cat_names=["Sex","Pclass","Embarked","Deck", "Title"],
    cont_names=['Age', 'SibSp', 'Parch', 'LogFare', 'Alone', 'TicketFreq', 'Family'],
    y_names="Survived", y_block = CategoryBlock(),
).dataloaders(path=".")

以下是每个参数的含义：

- 使用 `splits` 指定训练集和验证集的索引：

      splits=splits,
    
- 将字符串转为分类变量，用中位数填充数值列的缺失值，并对所有数值列进行归一化处理：
    
      procs = [Categorify, FillMissing, Normalize],
    
- 以下是类别型自变量：
    
      cat_names=["Sex","Pclass","Embarked","Deck", "Title"],
    
- 以下是连续型自变量：
    
      cont_names=['Age', 'SibSp', 'Parch', 'LogFare', 'Alone', 'TicketFreq', 'Family'],
    
- 以下是因变量：
    
      y_names="Survived",

- 因变量为分类变量（因此构建的是分类模型，而非回归模型）：

      y_block = CategoryBlock(),

## 训练模型

数据和模型共同构成一个 `Learner`。要创建一个，我们需要指定数据是什么（`dls`）、每个隐藏层的大小（`[10,10]`），以及我们希望在过程中打印的任何指标：

In [ ]:
learn = tabular_learner(dls, metrics=accuracy, layers=[10,10])

你会注意到，我们完全不需要费心去寻找一组能够正常训练的随机系数——这一切都是自动处理的。

fastai 还有一个实用的功能，可以帮我们推荐应该使用的学习率：

In [ ]:
learn.lr_find(suggest_funcs=(slide, valley))

两个彩色点都是学习率的合理选择。我会选择两者之间的某个值（0.03），并训练几个轮次：

In [ ]:
learn.fit(16, lr=0.03)

我们得到了与之前"从头开始"模型相似的准确率——这并不太令人惊讶，因为正如我们所讨论的，这个数据集太小且太简单，很难真正看出太大的差异。一个简单的线性模型已经做得相当不错了。但这没关系——这里的目标是向你展示如何开始深度学习，并理解它的真正工作原理，而最好的方式就是在小型且易于理解的数据集上进行。

## 提交到 Kaggle

fastai 的一个重要特性是，将数据变换和模型应用到新数据集所需的所有信息都存储在 learner 中。你可以调用 `export` 将其保存到文件，以便日后在生产环境中使用，也可以直接使用已训练好的模型对测试集进行预测。

要提交到 Kaggle，我们需要读入测试集，并对其进行与训练集相同的特征工程处理：

In [ ]:
tst_df = pd.read_csv(path/'test.csv')
tst_df['Fare'] = tst_df.Fare.fillna(0)
add_features(tst_df)

但我们不需要手动指定准备数据以进行建模所需的任何处理步骤，因为这些步骤都已保存在学习器中。要指定我们希望将相同的步骤应用于新数据集，请使用 `test_dl()` 方法：

In [ ]:
tst_dl = learn.dls.test_dl(tst_df)

现在我们可以使用 `get_preds` 来获取测试集的预测结果：

In [ ]:
preds,_ = learn.get_preds(dl=tst_dl)

最后，让我们像在之前的笔记本中所做的那样，创建一个提交的 CSV 文件……

In [ ]:
tst_df['Survived'] = (preds[:,1]>0.5).int()
sub_df = tst_df[['PassengerId','Survived']]
sub_df.to_csv('sub.csv', index=False)

……并检查它看起来是否合理：

In [ ]:
!head sub.csv

## 集成学习

由于现在创建模型变得非常简单，我们可以尝试一些更高级的建模方法。例如，我们可以从不同的随机初始点出发，训练五个独立的模型，然后对它们的预测结果取平均值。这是[集成学习](https://machinelearningmastery.com/tour-of-ensemble-learning-algorithms/)中最简单的一种方式——通过组合多个模型来生成比任何单一模型都更优的预测结果。

要构建我们的集成模型，首先将上面创建和训练模型的三个步骤复制下来，并应用到测试集上：

In [ ]:
def ensemble():
    learn = tabular_learner(dls, metrics=accuracy, layers=[10,10])
    with learn.no_bar(),learn.no_logging(): learn.fit(16, lr=0.03)
    return learn.get_preds(dl=tst_dl)[0]

现在我们运行五次，并将结果收集到一个列表中：

In [ ]:
learns = [ensemble() for _ in range(5)]

我们将这些预测叠加在一起，取其平均预测值：

In [ ]:
ens_preds = torch.stack(learns).mean(0)

最后，使用与之前相同的代码生成提交文件，在笔记本保存并运行后，我们可以将其提交到 Kaggle：

In [ ]:
tst_df['Survived'] = (ens_preds[:,1]>0.5).int()
sub_df = tst_df[['PassengerId','Survived']]
sub_df.to_csv('ens_sub.csv', index=False)

在撰写本文时，此提交结果处于竞赛所有参赛作品的前25%之列。

（本次竞赛中有很多参赛者使用了额外的外部数据，但我们将自己限制在仅使用所提供的数据范围内。如果使用外部数据，成绩可能会好很多。欢迎你尝试一下，看看效果如何。不过需要注意的是，你永远无法登上排行榜榜首，因为本次竞赛中有不少人通过从网上下载答案并直接提交的方式作弊。在真实竞赛中这是不可能的，因为答案并不公开，但在像这样的教程/练习竞赛中，没有任何机制能阻止作弊行为。所以，如果你已准备好迎接真正的挑战，不妨前往[竞赛页面](https://www.kaggle.com/competitions/)，开始参加一场真正的竞赛吧！）

## 最终想法

如你所见，使用 fastai 和 PyTorch 让整个过程比从头实现要轻松得多，但同时也隐藏了许多底层细节。所以，如果你只是单纯地使用框架，就很难真正理解背后发生了什么。而这种理解在调试和优化模型时往往非常有用。不过，在 Kaggle 比赛或实际项目中构建模型时，还是建议使用 fastai——否则你就白白错过了那些专门为你优化模型的大量研究成果，最终只会把更多时间耗费在调试和编写繁琐的样板代码上，而不是专注于解决真正的问题！

如果你觉得这个 notebook 对你有帮助，请记得点击顶部的小上箭头为它点赞——我很高兴看到大家觉得我的工作有价值，这也能帮助更多人发现它。（顺便提醒一下，点赞时请确认你看的是我的[原始 notebook](https://www.kaggle.com/jhoward/why-you-should-use-a-framework)，而不是你自己复制的版本，否则你的点赞不会被计入！）如果你有任何问题或想法，欢迎在下方留言——我会认真阅读每一条评论！

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文档已使用 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言的原始文档应视为权威来源。对于重要信息，建议进行专业的人工翻译。对于因使用本翻译而引起的任何误解或误读，我们概不负责。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
